In [2]:
!pip install duckdb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 18.1 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import duckdb
import pandas as pd

con = duckdb.connect('data/quant_signals.duckdb', read_only=True)

# What's in the database?
print('=== EQUITY DAILY ===')
print(con.execute('SELECT COUNT(*) as rows, MIN(trade_date) as earliest, MAX(trade_date) as latest FROM equity_daily').fetchdf())
print('\n=== F&O DAILY ===')
print(con.execute('SELECT COUNT(*) as rows, MIN(trade_date) as earliest, MAX(trade_date) as latest FROM fo_daily').fetchdf())

=== EQUITY DAILY ===
     rows   earliest     latest
0  895294 2023-01-02 2024-07-05

=== F&O DAILY ===
       rows   earliest     latest
0  17432444 2023-01-02 2024-06-04


In [4]:
# Delivery % for any stock over time
symbol = 'RELIANCE'  # change this to whatever you want

df = con.execute(f"""
    SELECT trade_date, close, volume, delivery_qty, delivery_pct
    FROM equity_daily
    WHERE symbol = '{symbol}' AND series = 'EQ'
    ORDER BY trade_date
""").fetchdf()

print(f'{symbol}: {len(df)} trading days')
df.tail(20)

RELIANCE: 395 trading days


,trade_date,close,volume,delivery_qty,delivery_pct
375,2024-06-10,2942.80,4625880,2334693,50.47
376,2024-06-11,2913.35,5887451,4001727,67.97
377,2024-06-12,2926.65,5040871,2648229,52.54
378,2024-06-13,2930.50,4590580,2387206,52.00
379,2024-06-14,2955.10,4078999,2250406,55.17
380,2024-06-17,2955.10,4078999,2250406,55.17
381,2024-06-18,2962.05,3598383,2137177,59.39
382,2024-06-19,2917.30,4362937,2395420,54.90
383,2024-06-20,2947.40,8056888,3570538,44.32
384,2024-06-21,2908.40,15585180,8287080,53.17


In [5]:
# Unusual delivery spikes — stocks where delivery % is >2 std devs above their own 20-day mean
# These are potential "informed accumulation" signals

spikes = con.execute("""
    WITH stats AS (
        SELECT 
            symbol, trade_date, close, volume, delivery_pct,
            AVG(delivery_pct) OVER (PARTITION BY symbol ORDER BY trade_date ROWS BETWEEN 20 PRECEDING AND 1 PRECEDING) as avg_del_20d,
            STDDEV(delivery_pct) OVER (PARTITION BY symbol ORDER BY trade_date ROWS BETWEEN 20 PRECEDING AND 1 PRECEDING) as std_del_20d
        FROM equity_daily
        WHERE series = 'EQ' AND volume > 100000
    )
    SELECT symbol, trade_date, close, volume, delivery_pct, avg_del_20d, std_del_20d,
           (delivery_pct - avg_del_20d) / NULLIF(std_del_20d, 0) as z_score
    FROM stats
    WHERE std_del_20d > 0 AND (delivery_pct - avg_del_20d) / std_del_20d > 2.0
    ORDER BY trade_date DESC, z_score DESC
    LIMIT 30
""").fetchdf()

spikes

,symbol,trade_date,close,volume,delivery_pct,avg_del_20d,std_del_20d,z_score
0,ABCAPITAL,2024-07-05,236.01,4524859,65.98,33.6325,4.171423,7.754549
1,HEG,2024-07-05,2264.60,1373947,81.86,33.8660,8.772275,5.471101
2,MARICO,2024-07-05,615.35,6070845,84.68,54.5820,7.206554,4.176476
3,AMBUJACEM,2024-07-05,686.00,4592446,69.99,42.7000,7.528843,3.624727
4,TIMETECHNO,2024-07-05,327.35,524272,56.74,31.5025,7.090676,3.559252
5,HILTON,2024-07-05,88.25,518498,65.33,36.6855,8.048070,3.559176
6,MASTEK,2024-07-05,2838.70,126884,60.58,33.1990,8.904887,3.074829
7,ELECTCAST,2024-07-05,189.05,1641255,66.67,54.4825,3.973038,3.067552
8,TPLPLASTEH,2024-07-05,98.08,924840,50.56,24.9185,8.516874,3.010670
9,JNKINDIA,2024-07-05,882.65,377637,76.12,45.9990,10.039533,3.000239


In [6]:
# NIFTY futures OI buildup — are institutions adding long positions?

nifty_fut = con.execute("""
    SELECT trade_date, expiry, close, oi, oi_change,
           SUM(oi_change) OVER (ORDER BY trade_date ROWS BETWEEN 4 PRECEDING AND CURRENT ROW) as oi_change_5d
    FROM fo_daily
    WHERE symbol = 'NIFTY' AND instrument = 'FUTIDX'
      AND expiry = (
          SELECT MIN(expiry) FROM fo_daily 
          WHERE symbol = 'NIFTY' AND instrument = 'FUTIDX' AND expiry > trade_date
      )
    ORDER BY trade_date DESC
    LIMIT 30
""").fetchdf()

nifty_fut

,trade_date,expiry,close,oi,oi_change,oi_change_5d
0,2023-01-25,2023-01-25,17888.50,2732250,-4053900,-8611650.0
1,2023-01-24,2023-01-25,18128.25,6786150,-2045150,-4768050.0
2,2023-01-23,2023-01-25,18148.15,8831300,-1582650,-2939700.0
3,2023-01-20,2023-01-25,18055.80,10413950,-222300,-1466850.0
4,2023-01-19,2023-01-25,18113.15,10636250,-707650,-1019650.0
5,2023-01-18,2023-01-25,18199.15,11343900,-210300,-148900.0
6,2023-01-17,2023-01-25,18088.80,11554200,-216800,-86150.0
7,2023-01-16,2023-01-25,17941.75,11771000,-109800,511100.0
8,2023-01-13,2023-01-25,18025.25,11880800,224900,359250.0
9,2023-01-12,2023-01-25,17918.65,11655900,163100,498300.0


In [7]:
# Single stock F&O — OI buildup on a specific name
symbol = 'HDFCBANK'  # change this

stock_fut = con.execute(f"""
    SELECT trade_date, expiry, close, oi, oi_change
    FROM fo_daily
    WHERE symbol = '{symbol}' AND instrument = 'FUTSTK'
    ORDER BY trade_date DESC, expiry ASC
    LIMIT 20
""").fetchdf()

stock_fut

,trade_date,expiry,close,oi,oi_change
0,2024-06-04,2024-06-27,1487.85,184465050,-6525750
1,2024-06-04,2024-07-25,1498.50,3758150,1613700
2,2024-06-04,2024-08-29,1511.60,432850,259050
3,2024-06-03,2024-06-27,1582.05,190990800,-3880250
4,2024-06-03,2024-07-25,1593.35,2144450,-154000
5,2024-06-03,2024-08-29,1605.50,173800,115500
6,2024-05-31,2024-06-27,1539.75,194871050,-2007500
7,2024-05-31,2024-07-25,1550.80,2298450,4950
8,2024-05-31,2024-08-29,1559.95,58300,58300
9,2024-05-30,2024-05-30,1514.00,5546750,-19882500


In [8]:
# Top call OI buildup — who's betting on upside?

call_oi = con.execute("""
    WITH latest AS (SELECT MAX(trade_date) as d FROM fo_daily)
    SELECT f.symbol, f.strike, f.expiry, f.oi, f.oi_change, f.close
    FROM fo_daily f, latest l
    WHERE f.trade_date = l.d 
      AND f.instrument = 'OPTSTK' 
      AND f.option_type = 'CE'
      AND f.oi_change > 0
    ORDER BY f.oi_change DESC
    LIMIT 20
""").fetchdf()

call_oi

,symbol,strike,expiry,oi,oi_change,close
0,IDEA,14.0,2024-06-27,121600000,62520000,0.60
1,IDEA,15.0,2024-06-27,163520000,55720000,0.35
2,IDEA,16.0,2024-06-27,210600000,51200000,0.25
3,IDEA,17.0,2024-06-27,139520000,30120000,0.15
4,IDEA,18.0,2024-06-27,138120000,24480000,0.15
5,IDEA,13.0,2024-06-27,41160000,17880000,1.00
6,IDEA,20.0,2024-06-27,103600000,13600000,0.10
7,IDFCFIRSTB,75.0,2024-06-27,12922500,7860000,2.25
8,IDEA,12.0,2024-06-27,9880000,6560000,1.70
9,BEL,300.0,2024-06-27,11628000,5719950,9.25


In [10]:
# Load more data as backfill continues (run this cell to refresh)
import sys
sys.path.insert(0, '.')
from src.ingestion.load_to_duckdb import load_all_available

# Close read-only connection, load new data, reopen
con.close()
load_all_available()
con = duckdb.connect('data/quant_signals.duckdb', read_only=True)
print('\nRefreshed. Current state:')
print(con.execute('SELECT COUNT(*) as rows, MIN(trade_date) as earliest, MAX(trade_date) as latest FROM equity_daily').fetchdf())

  Equity 2024-07-08: 2526 rows
  Equity 2024-07-09: 2488 rows
  Equity 2024-07-10: 2492 rows
  Equity 2024-07-11: 2492 rows
  Equity 2024-07-12: 2497 rows
  Equity 2024-07-15: 2538 rows
  Equity 2024-07-16: 2500 rows
  Equity 2024-07-17: 2500 rows
  Equity 2024-07-18: 2500 rows
  Equity 2024-07-19: 2493 rows
  Equity 2024-07-22: 2536 rows
  Equity 2024-07-23: 2493 rows
  Equity 2024-07-24: 2491 rows
  Equity 2024-07-25: 2495 rows
  Equity 2024-07-26: 2500 rows
  Equity 2024-07-29: 2541 rows
  Equity 2024-07-30: 2502 rows
  Equity 2024-07-31: 2511 rows
  Equity 2024-08-01: 2524 rows
  Equity 2024-08-02: 2516 rows
  Equity 2024-08-05: 2561 rows
  Equity 2024-08-06: 2526 rows
  Equity 2024-08-07: 2525 rows
  Equity 2024-08-08: 2518 rows
  Equity 2024-08-09: 2523 rows
  Equity 2024-08-12: 2565 rows
  Equity 2024-08-13: 2529 rows
  Equity 2024-08-14: 2512 rows
  Equity 2024-08-15: 2512 rows
  Equity 2024-08-16: 2525 rows
  Equity 2024-08-19: 2558 rows
  Equity 2024-08-20: 2531 rows
  Equity

BinderException: Binder Error: table fo_daily has 15 columns but 1 values were supplied